# Human Connectome Project 7T fMRI Retinotopy Dataset

This notebook shows how to automate the download and management of **Human Connectome Project (HCP) 7T fMRI dataset**. The data involved ultra-high 7 Tesla fMRI retinotopic mapping data in 181 healthy adults, with additional movie-watching, and resting-state runs (1.6-mm resolution and repetition time of 1s).


The notebook covers five main stages: configuring the environment and AWS credentials for S3 access, loading validated subject IDs with complete retinotopy data, defining task parameters for retinotopy (RETBAR 1-2), movie-watching (MOVIE 1-4), and resting-state (REST 1-4) scans, checking the download status of existing data, and performing batch downloads from the [HCP OpenAccess S3 bucket](https://wiki.humanconnectome.org/docs/How%20to%20Get%20Access%20to%20the%20HCP%20OpenAccess%20Amazon%20S3%20Bucket.html). Each subject requires approximately 2-3 GB of storage across 10 scan runs. Note that HCP database access (free registration) is required, and AWS credentials should be configured before running the download scripts.


Benson, N. C., Jamison, K. W., Arcaro, M. J., Vu, A. T., Glasser, M. F., Coalson, T. S., Van Essen, D. V., Yacoub, E., Ugurbil, K., Winawer, J., & Kay, K. (2018). **The Human Connectome Project 7 Tesla retinotopy dataset: Description and population receptive field analysis**. Journal of Vision, 18(13), 23. https://doi.org/10.1167/18.13.23


## Environment setup & dependencies

Configure AWS credentials and import necessary libraries for CIFTI file handling and S3 data access.

In [ ]:
import os
from nibabel import cifti2
import numpy as np
import itertools
import nibabel as nib
import argparse
import boto3
import sys
import pickle


aws_access_key_id='xxxx...'
aws_secret_access_key='xxxx...'


## Subject ID configuration

List of 184 validated HCP subject IDs with complete 7T retinotopy data.

To get the 7T `subj_id.txt` :

```shell
100610
102311
102816
104416
105923
108323
109123
111312
111514
114823
115017
115825
116726
118225
125525
126426
128935
130518
131217
131722
134627
134829
135124
137128
140117
144226
145834
146129
146432
146735
146937
148133
150423
155938
156334
157336
158035
158136
159239
162935
164131
164636
165436
167036
167440
169343
169747
171633
172130
173334
175237
176542
177140
177645
177746
178142
178243
178647
180533
181232
181636
182436
182739
185442
186949
187345
191033
191336
191841
192439
192641
193845
195041
196144
197348
198653
199655
200210
200311
200614
201515
203418
204521
205220
209228
212419
214019
214524
221319
233326
239136
246133
249947
251833
257845
263436
283543
318637
320826
330324
346137
352738
360030
365343
380036
381038
389357
393247
395756
397760
406836
412528
429040
436845
463040
467351
473952
525541
536647
541943
547046
550439
552241
562345
572045
573249
581450
585256
601127
617748
627549
638049
654552
671855
680957
690152
706040
724446
725751
732243
745555
751550
757764
765864
770352
771354
782561
783462
789373
814649
818859
825048
826353
833249
859671
861456
871762
872764
878776
878877
898176
899885
901139
905147
910241
926862
927359
942658
951457
958976
966975
995174
```

## Task & path definition

Define task IDs for all scan types and configure local storage paths.

In [3]:
fileobj     = open('subj_id.txt')
data        = fileobj.read()
subject_ID  = data.split('\n')
print('Subjects : ', subject_ID)

task_ID_1  = ['tfMRI_RETBAR1_7T_AP',
              'tfMRI_RETBAR2_7T_PA'
              ]

task_ID_2  = ['tfMRI_MOVIE1_7T_AP',
              'tfMRI_MOVIE2_7T_PA',
              'tfMRI_MOVIE3_7T_PA',
              'tfMRI_MOVIE4_7T_AP', 
             ]

task_ID_3  = ['rfMRI_REST1_7T_PA',
              'rfMRI_REST2_7T_AP',
              'rfMRI_REST3_7T_PA',
              'rfMRI_REST4_7T_AP'
             ]

task_ID = task_ID_1 + task_ID_2 + task_ID_3
print('Number of tasks : ',len(task_ID))

temp_root = '/scratch/nicogravel/HCP_data/temp/'
file      = '_Atlas_1.6mm_MSMAll_hp2000_clean.dtseries.nii'

Subjects :  ['100610', '102311', '102816', '104416', '105923', '108323', '109123', '111312', '111514', '114823', '115017', '115825', '116726', '118225', '125525', '126426', '128935', '130518', '131217', '131722', '134627', '134829', '135124', '137128', '140117', '144226', '145834', '146129', '146432', '146735', '146937', '148133', '150423', '155938', '156334', '157336', '158035', '158136', '159239', '162935', '164131', '164636', '165436', '167036', '167440', '169343', '169747', '171633', '172130', '173334', '175237', '176542', '177140', '177645', '177746', '178142', '178243', '178647', '180533', '181232', '181636', '182436', '182739', '185442', '186949', '187345', '191033', '191336', '191841', '192439', '192641', '193845', '195041', '196144', '197348', '198653', '199655', '200210', '200311', '200614', '201515', '203418', '204521', '205220', '209228', '212419', '214019', '214524', '221319', '233326', '239136', '246133', '249947', '251833', '257845', '263436', '283543', '318637', '320826

## Download status check

Scan local directory to identify already-downloaded subjects and calculate storage requirements.

In [4]:
n_subj = 0
x_subj = 0
for subj in range(len(subject_ID)):
    isdir = os.path.isdir(temp_root + subject_ID[subj] + '/')   
    #print(isdir)   
    if isdir == True: 
        folder = temp_root + subject_ID[subj]          
        # get size
        size = 0
        for ele in os.scandir(folder):
            size+=os.path.getsize(ele)
        print('subject: ', subject_ID[subj], ', size(MB): ',  size/1024**2)  
        n_subj = + n_subj + 1
    if isdir == False: 
        x_subj = + x_subj + 1
print('Subject downloaded : ',n_subj, ',  Subject to download : ', x_subj)


subject:  100610 , size(MB):  5123.201560974121
subject:  102311 , size(MB):  5123.201560974121
subject:  102816 , size(MB):  5123.201560974121
subject:  104416 , size(MB):  5123.201560974121
subject:  105923 , size(MB):  5123.201560974121
subject:  108323 , size(MB):  5123.201560974121
subject:  109123 , size(MB):  5123.201560974121
subject:  111312 , size(MB):  4536.369209289551
subject:  111514 , size(MB):  5120.202827453613
subject:  114823 , size(MB):  5120.202827453613
subject:  115017 , size(MB):  5120.202827453613
subject:  115825 , size(MB):  5120.202827453613
subject:  116726 , size(MB):  5120.202827453613
subject:  118225 , size(MB):  5120.202827453613
subject:  125525 , size(MB):  5120.202827453613
subject:  126426 , size(MB):  5120.202827453613
subject:  128935 , size(MB):  5120.202827453613
subject:  130518 , size(MB):  5120.202827453613
subject:  131217 , size(MB):  5120.202827453613
subject:  131722 , size(MB):  5120.202827453613
subject:  134627 , size(MB):  5120.20282

## Batch download

Download CIFTI timeseries files from HCP OpenAccess bucket for all missing subjects and task runs.

In [4]:
n_subj = 0
x_subj = 0

for subj in range(len(subject_ID)):
    isdir = os.path.isdir(temp_root + subject_ID[subj] + '/')   
  
    
    if isdir == True: 
        folder = temp_root + subject_ID[subj]          
        # get size
        size = 0
        for ele in os.scandir(folder):
            size+=os.path.getsize(ele)
        print('subject: ', subject_ID[subj], ', size(MB): ',  size/1024**2)  
        n_subj = + n_subj + 1
        
    if isdir == False: 
        os.mkdir(temp_root + subject_ID[subj])
        for run in range(len(task_ID)):
            print('Processing data for subject :', subject_ID[subj], 'task :', task_ID[run])
            dbhcp = 'HCP_1200/' + subject_ID[subj] + '/MNINonLinear/Results/'
            s3_client = boto3.client('s3',
                                    aws_access_key_id = aws_access_key_id,
                                    aws_secret_access_key = aws_secret_access_key)
            s3_bucket_name = 'hcp-openaccess'
            s3_path = dbhcp + task_ID[run] + '/' + task_ID[run] + file
            print(s3_path)
            temp_save_path = temp_root + subject_ID[subj] + '/' + task_ID[run] + file
            print(temp_save_path)
            print('Downloading...')
            s3_client.download_file(Bucket=s3_bucket_name, Key=s3_path, Filename=temp_save_path)
            sub_file = temp_save_path

subject:  100610 , size(MB):  5123.201560974121
subject:  102311 , size(MB):  5123.201560974121
subject:  102816 , size(MB):  5123.201560974121
subject:  104416 , size(MB):  5123.201560974121
subject:  105923 , size(MB):  5123.201560974121
subject:  108323 , size(MB):  5123.201560974121
subject:  109123 , size(MB):  5123.201560974121
subject:  111312 , size(MB):  4536.369209289551
subject:  111514 , size(MB):  5120.202827453613
subject:  114823 , size(MB):  5120.202827453613
subject:  115017 , size(MB):  5120.202827453613
subject:  115825 , size(MB):  5120.202827453613
subject:  116726 , size(MB):  5120.202827453613
subject:  118225 , size(MB):  5120.202827453613
subject:  125525 , size(MB):  5120.202827453613
subject:  126426 , size(MB):  5120.202827453613
subject:  128935 , size(MB):  5120.202827453613
subject:  130518 , size(MB):  5120.202827453613
subject:  131217 , size(MB):  5120.202827453613
subject:  131722 , size(MB):  5120.202827453613
subject:  134627 , size(MB):  5120.20282

Processing data for subject : 826353 task : tfMRI_MOVIE3_7T_PA
HCP_1200/826353/MNINonLinear/Results/tfMRI_MOVIE3_7T_PA/tfMRI_MOVIE3_7T_PA_Atlas_1.6mm_MSMAll_hp2000_clean.dtseries.nii
/scratch/nicogravel/HCP_data/temp/826353/tfMRI_MOVIE3_7T_PA_Atlas_1.6mm_MSMAll_hp2000_clean.dtseries.nii
Downloading...
Processing data for subject : 826353 task : tfMRI_MOVIE4_7T_AP
HCP_1200/826353/MNINonLinear/Results/tfMRI_MOVIE4_7T_AP/tfMRI_MOVIE4_7T_AP_Atlas_1.6mm_MSMAll_hp2000_clean.dtseries.nii
/scratch/nicogravel/HCP_data/temp/826353/tfMRI_MOVIE4_7T_AP_Atlas_1.6mm_MSMAll_hp2000_clean.dtseries.nii
Downloading...
Processing data for subject : 826353 task : rfMRI_REST1_7T_PA
HCP_1200/826353/MNINonLinear/Results/rfMRI_REST1_7T_PA/rfMRI_REST1_7T_PA_Atlas_1.6mm_MSMAll_hp2000_clean.dtseries.nii
/scratch/nicogravel/HCP_data/temp/826353/rfMRI_REST1_7T_PA_Atlas_1.6mm_MSMAll_hp2000_clean.dtseries.nii
Downloading...
Processing data for subject : 826353 task : rfMRI_REST2_7T_AP
HCP_1200/826353/MNINonLinear/Resu

HCP_1200/861456/MNINonLinear/Results/tfMRI_RETBAR2_7T_PA/tfMRI_RETBAR2_7T_PA_Atlas_1.6mm_MSMAll_hp2000_clean.dtseries.nii
/scratch/nicogravel/HCP_data/temp/861456/tfMRI_RETBAR2_7T_PA_Atlas_1.6mm_MSMAll_hp2000_clean.dtseries.nii
Downloading...
Processing data for subject : 861456 task : tfMRI_MOVIE1_7T_AP
HCP_1200/861456/MNINonLinear/Results/tfMRI_MOVIE1_7T_AP/tfMRI_MOVIE1_7T_AP_Atlas_1.6mm_MSMAll_hp2000_clean.dtseries.nii
/scratch/nicogravel/HCP_data/temp/861456/tfMRI_MOVIE1_7T_AP_Atlas_1.6mm_MSMAll_hp2000_clean.dtseries.nii
Downloading...
Processing data for subject : 861456 task : tfMRI_MOVIE2_7T_PA
HCP_1200/861456/MNINonLinear/Results/tfMRI_MOVIE2_7T_PA/tfMRI_MOVIE2_7T_PA_Atlas_1.6mm_MSMAll_hp2000_clean.dtseries.nii
/scratch/nicogravel/HCP_data/temp/861456/tfMRI_MOVIE2_7T_PA_Atlas_1.6mm_MSMAll_hp2000_clean.dtseries.nii
Downloading...
Processing data for subject : 861456 task : tfMRI_MOVIE3_7T_PA
HCP_1200/861456/MNINonLinear/Results/tfMRI_MOVIE3_7T_PA/tfMRI_MOVIE3_7T_PA_Atlas_1.6mm_M

Processing data for subject : 872764 task : rfMRI_REST4_7T_AP
HCP_1200/872764/MNINonLinear/Results/rfMRI_REST4_7T_AP/rfMRI_REST4_7T_AP_Atlas_1.6mm_MSMAll_hp2000_clean.dtseries.nii
/scratch/nicogravel/HCP_data/temp/872764/rfMRI_REST4_7T_AP_Atlas_1.6mm_MSMAll_hp2000_clean.dtseries.nii
Downloading...
Processing data for subject : 878776 task : tfMRI_RETBAR1_7T_AP
HCP_1200/878776/MNINonLinear/Results/tfMRI_RETBAR1_7T_AP/tfMRI_RETBAR1_7T_AP_Atlas_1.6mm_MSMAll_hp2000_clean.dtseries.nii
/scratch/nicogravel/HCP_data/temp/878776/tfMRI_RETBAR1_7T_AP_Atlas_1.6mm_MSMAll_hp2000_clean.dtseries.nii
Downloading...
Processing data for subject : 878776 task : tfMRI_RETBAR2_7T_PA
HCP_1200/878776/MNINonLinear/Results/tfMRI_RETBAR2_7T_PA/tfMRI_RETBAR2_7T_PA_Atlas_1.6mm_MSMAll_hp2000_clean.dtseries.nii
/scratch/nicogravel/HCP_data/temp/878776/tfMRI_RETBAR2_7T_PA_Atlas_1.6mm_MSMAll_hp2000_clean.dtseries.nii
Downloading...
Processing data for subject : 878776 task : tfMRI_MOVIE1_7T_AP
HCP_1200/878776/MNINonLi